In [2]:
import pandas as pd

df = pd.read_csv('../../datasets/US_with_labels.csv')
df.head()

,spotify_id,name,artists,daily_rank,daily_movement,weekly_movement,country,snapshot_date,popularity,is_explicit,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,average_song
0,2CGNAOSuO1MEFCbBRgUzjd,luther (with sza),"Kendrick Lamar, SZA",1,0,0,US,2025-02-17,90,False,...,-7.546,1,0.1250,0.2510,0.000000,0.2480,0.576,138.008,4,About_Average
1,6AI3ezQ4o3HUoP6Dhudph3,Not Like Us,Kendrick Lamar,2,0,1,US,2025-02-17,92,True,...,-7.001,1,0.0776,0.0107,0.000000,0.1410,0.214,101.061,4,Higher
2,0aB0v4027ukVziUGwVGYpG,tv off (feat. lefty gunplay),"Kendrick Lamar, Lefty Gunplay",3,0,-1,US,2025-02-17,92,True,...,-6.679,0,0.2630,0.0837,0.000000,0.4230,0.548,100.036,4,Higher
3,3GCdLUSnKSMJhs4Tj6CV3s,All The Stars (with SZA),"Kendrick Lamar, SZA",4,0,12,US,2025-02-17,90,True,...,-4.946,1,0.0599,0.0612,0.000195,0.0926,0.557,96.782,4,About_Average
4,0nj9Bq5sHDiTxSHunhgkFb,squabble up,Kendrick Lamar,5,0,4,US,2025-02-17,89,True,...,-5.568,1,0.1980,0.0206,0.000000,0.0783,0.711,103.921,4,Lower


In [3]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

#a = df[df["name"] == "APT."]
X = df.drop(columns=["spotify_id", "name", "artists", "snapshot_date", "country", "album_name", "album_release_date", "popularity"], axis=1, inplace=False)
y = df["popularity"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.columns)

scaler = MinMaxScaler()
encode = OneHotEncoder()


Index(['daily_rank', 'daily_movement', 'weekly_movement', 'is_explicit',
       'duration_ms', 'danceability', 'energy', 'key', 'loudness', 'mode',
       'speechiness', 'acousticness', 'instrumentalness', 'liveness',
       'valence', 'tempo', 'time_signature', 'average_song'],
      dtype='object')


In [4]:
from sklearn.compose import make_column_transformer
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score
from sklearn.metrics import root_mean_squared_error


preprocessing = make_column_transformer((encode, ['average_song']), (scaler, ['key', 'daily_rank', 'daily_movement', 'weekly_movement',
       'is_explicit', 'duration_ms', 'danceability', 'energy', 'key',
       'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness',
       'liveness', 'valence', 'tempo', 'time_signature'] ), remainder='passthrough')

pipeline = make_pipeline(
    preprocessing,
    RandomForestRegressor(n_estimators=200, min_samples_split=2, min_samples_leaf=2, max_features=0.8, max_depth=30, n_jobs=4)
)
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

# https://scikit-learn.org/stable/auto_examples/ensemble/plot_forest_hist_grad_boosting_comparison.html


In [5]:
print("Random Forests Regression")
print("Mean Absolute Error: ", mean_absolute_error(y_test, y_pred))
print("Mean Squared Error: ", mean_squared_error(y_test, y_pred))
print("Root Mean Squared Error: ", root_mean_squared_error(y_test, y_pred))
print("R2 Score: ", r2_score(y_test, y_pred))

for i in range(10):
    print(f"Predicted: {y_pred[i]}, Actual: {y_test.iloc[i]}")


Random Forests Regression
Mean Absolute Error:  2.883645198811399
Mean Squared Error:  59.70698331989656
Root Mean Squared Error:  7.727029398151437
R2 Score:  0.5950795021312987
Predicted: 83.59331349206349, Actual: 82
Predicted: 90.58053787094546, Actual: 90
Predicted: 94.22521972666591, Actual: 90
Predicted: 86.83901785714285, Actual: 86
Predicted: 88.55374585137083, Actual: 89
Predicted: 88.82967919799499, Actual: 89
Predicted: 84.4997665233161, Actual: 86
Predicted: 93.91503535353537, Actual: 94
Predicted: 83.07988474025973, Actual: 85
Predicted: 83.85558730158729, Actual: 84


In [6]:
param_distributions = {
    'randomforestregressor__n_estimators': [50, 100, 200, 300],
    'randomforestregressor__max_depth': [5, 10, 20, 30, None],
    'randomforestregressor__min_samples_split': [2, 5, 10],
    'randomforestregressor__min_samples_leaf': [1, 2, 4],
    'randomforestregressor__max_features': ['sqrt', 'log2', 0.8]
}

from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import KFold

RFRGrid = RandomizedSearchCV(
    pipeline,
    param_distributions=param_distributions,
    scoring="neg_mean_squared_error",
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    n_iter=20,
    random_state=42,
    verbose=1
)

RFRGrid.fit(X_train, y_train)
print("Random Forests Regression with Grid Search")
print(RFRGrid.best_params_)
print(RFRGrid.best_score_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Random Forests Regression with Grid Search
{'randomforestregressor__n_estimators': 100, 'randomforestregressor__min_samples_split': 2, 'randomforestregressor__min_samples_leaf': 1, 'randomforestregressor__max_features': 'log2', 'randomforestregressor__max_depth': None}
-62.72244408931217
